In [1]:
import pandas as pd
import numpy as np
import tensorflow as tf
from sklearn.preprocessing import StandardScaler

df = pd.read_csv('data/jena_cleaned.csv', index_col='Date Time', parse_dates=True)

print(df.shape)
df.head()

I0000 00:00:1780400122.691077    4317 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1780400123.866387    4317 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1780400132.232731    4317 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


(420551, 14)


,p (mbar),T (degC),Tpot (K),Tdew (degC),rh (%),VPmax (mbar),VPact (mbar),VPdef (mbar),sh (g/kg),H2OC (mmol/mol),rho (g/m**3),wv (m/s),max. wv (m/s),wd (deg)
Date Time,,,,,,,,,,,,,,
2009-01-01 00:10:00,996.52,-8.02,265.40,-8.90,93.3,3.33,3.11,0.22,1.94,3.12,1307.75,1.03,1.75,152.3
2009-01-01 00:20:00,996.57,-8.41,265.01,-9.28,93.4,3.23,3.02,0.21,1.89,3.03,1309.80,0.72,1.50,136.1
2009-01-01 00:30:00,996.53,-8.51,264.91,-9.31,93.9,3.21,3.01,0.20,1.88,3.02,1310.24,0.19,0.63,171.6
2009-01-01 00:40:00,996.51,-8.31,265.12,-9.07,94.2,3.26,3.07,0.19,1.92,3.08,1309.19,0.34,0.50,198.0
2009-01-01 00:50:00,996.51,-8.27,265.15,-9.04,94.1,3.27,3.08,0.19,1.92,3.09,1309.00,0.32,0.63,214.3


In [2]:
# Select features
features = ['T (degC)', 'p (mbar)', 'Tdew (degC)', 'rh (%)', 
            'VPact (mbar)', 'VPdef (mbar)', 'wv (m/s)', 'wd (deg)']

df = df[features]

# Drop rows with NaN
df = df.dropna()

print(f"Shape after feature selection and NaN removal: {df.shape}")

Shape after feature selection and NaN removal: (420533, 8)


In [3]:
# Cyclical encoding for wind direction
df['wd_sin'] = np.sin(2 * np.pi * df['wd (deg)'] / 360)
df['wd_cos'] = np.cos(2 * np.pi * df['wd (deg)'] / 360)
df = df.drop(columns=['wd (deg)'])

# Cyclical encoding for time features
df['hour_sin'] = np.sin(2 * np.pi * df.index.hour / 24)
df['hour_cos'] = np.cos(2 * np.pi * df.index.hour / 24)
df['doy_sin'] = np.sin(2 * np.pi * df.index.dayofyear / 365)
df['doy_cos'] = np.cos(2 * np.pi * df.index.dayofyear / 365)

print(f"Shape after encoding: {df.shape}")
print(df.head())

Shape after encoding: (420533, 13)
                     T (degC)  p (mbar)  Tdew (degC)  rh (%)  VPact (mbar)  \
Date Time                                                                    
2009-01-01 00:10:00     -8.02    996.52        -8.90    93.3          3.11   
2009-01-01 00:20:00     -8.41    996.57        -9.28    93.4          3.02   
2009-01-01 00:30:00     -8.51    996.53        -9.31    93.9          3.01   
2009-01-01 00:40:00     -8.31    996.51        -9.07    94.2          3.07   
2009-01-01 00:50:00     -8.27    996.51        -9.04    94.1          3.08   

                     VPdef (mbar)  wv (m/s)    wd_sin    wd_cos  hour_sin  \
Date Time                                                                   
2009-01-01 00:10:00          0.22      1.03  0.464842 -0.885394       0.0   
2009-01-01 00:20:00          0.21      0.72  0.693402 -0.720551       0.0   
2009-01-01 00:30:00          0.20      0.19  0.146083 -0.989272       0.0   
2009-01-01 00:40:00          0.19

In [4]:
# chronological split
n = len(df)

train_size = int(n * 0.70) 
val_size   = int(n * 0.15) 
test_size  = n - train_size - val_size  

train_df = df[:train_size]
val_df   = df[train_size:train_size + val_size]
test_df  = df[train_size + val_size:]

print(f"Total rows: {n}")
print(f"Train size: {train_size} ({train_size/n*100:.1f}%)")
print(f"Val size:   {val_size} ({val_size/n*100:.1f}%)")
print(f"Test size:  {test_size} ({test_size/n*100:.1f}%)")
print()
print(f"Train: {train_df.index[0]} to {train_df.index[-1]}")
print(f"Val:   {val_df.index[0]} to {val_df.index[-1]}")
print(f"Test:  {test_df.index[0]} to {test_df.index[-1]}")

Total rows: 420533
Train size: 294373 (70.0%)
Val size:   63079 (15.0%)
Test size:  63081 (15.0%)

Train: 2009-01-01 00:10:00 to 2014-08-05 00:20:00
Val:   2014-08-05 00:30:00 to 2015-10-17 20:20:00
Test:  2015-10-17 20:30:00 to 2017-01-01 00:00:00


In [5]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

train_scaled = scaler.fit_transform(train_df)
val_scaled   = scaler.transform(val_df)
test_scaled  = scaler.transform(test_df)

print(f"Train mean: {train_scaled.mean():.4f}")
print(f"Train std: {train_scaled.std():.4f}")

Train mean: 0.0000
Train std: 1.0000


In [7]:
def create_sequences(data, window_size, forecast_horizon, target_col=0):
    X, y = [], []
    for i in range(len(data) - window_size - forecast_horizon):
        X.append(data[i:i+window_size])
        y.append(data[i+window_size+forecast_horizon, target_col])
    return np.array(X), np.array(y)

# steps = hours x (60minutes/10min) = hours x 6 
window_size = 72 # 12 hours = 12 x 6 = 72 steps
forecast_horizon = 144 # 24 hours = 24 x 6 = 144 steps

X_train, y_train = create_sequences(train_scaled, window_size, forecast_horizon)
X_val, y_val = create_sequences(val_scaled, window_size, forecast_horizon)
X_test, y_test = create_sequences(test_scaled, window_size, forecast_horizon)

print(f"X_train shape: {X_train.shape}, y_train shape: {y_train.shape}")
print(f"X_val shape: {X_val.shape}, y_val shape: {y_val.shape}")
print(f"X_test shape: {X_test.shape}, y_test shape: {y_test.shape}")

X_train shape: (294157, 72, 13), y_train shape: (294157,)
X_val shape: (62863, 72, 13), y_val shape: (62863,)
X_test shape: (62865, 72, 13), y_test shape: (62865,)
